In [ ]:
import pandas as pd
import os
import glob

# Set base directory relative to notebook location
BASE_DIR = os.path.dirname(os.path.abspath('02_eda.ipynb'))
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
CLEAN_DIR = os.path.join(BASE_DIR, 'data', 'cleaned')
FIG_DIR = os.path.join(BASE_DIR, 'outputs', 'figures')

# Load and stack all 14 state registration files
files = glob.glob(os.path.join(RAW_DIR, '*EV_Registrations*.csv'))
print(f"Found {len(files)} state files:")
for f in files:
    print(" ", os.path.basename(f))

In [ ]:
# Load and stack all 14 files
dfs = []
for f in files:
    df_temp = pd.read_csv(f, low_memory=False)
    dfs.append(df_temp)

df_reg = pd.concat(dfs, ignore_index=True)

print(f"Total rows: {df_reg.shape[0]:,}")
print(f"Columns: {df_reg.shape[1]}")
print(f"\nColumn names:\n{df_reg.columns.tolist()}")
print(f"\nSample:\n{df_reg.head()}")

In [ ]:
# Filter to BEV only and latest snapshot per state
df_bev = df_reg[df_reg['Drivetrain Type'] == 'BEV'].copy()

# Convert Registration Date to datetime and extract quarter
df_bev['Registration Date'] = pd.to_datetime(df_bev['Registration Date'])
df_bev['quarter'] = df_bev['Registration Date'].dt.to_period('Q')

# Flag Tesla vs all other makes
df_bev['is_tesla'] = df_bev['Vehicle Make'].str.upper() == 'TESLA'

# Aggregate: total BEV count and Tesla count by state + quarter
df_agg = df_bev.groupby(['State', 'quarter', 'is_tesla'])['Vehicle Count'].sum().reset_index()

# Pivot so Tesla and non-Tesla are columns
df_pivot = df_agg.pivot_table(
    index=['State', 'quarter'],
    columns='is_tesla',
    values='Vehicle Count',
    aggfunc='sum'
).reset_index()

df_pivot.columns = ['state', 'quarter', 'non_tesla_bev', 'tesla_bev']
df_pivot['total_bev'] = df_pivot['tesla_bev'] + df_pivot['non_tesla_bev']
df_pivot['tesla_share'] = df_pivot['tesla_bev'] / df_pivot['total_bev']

print(df_pivot.shape)
print(df_pivot.head(10))

df_pivot.to_csv(os.path.join(CLEAN_DIR, 'ev_registrations_clean.csv'), index=False)
print("Saved.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os

# Filter to 2020 onwards for relevant analysis period
df_recent = df_pivot[df_pivot['quarter'].astype(str) >= '2020Q1'].copy()

# Aggregate across all states by quarter
df_national = df_recent.groupby('quarter')[['tesla_bev', 'total_bev']].sum().reset_index()
df_national['tesla_share'] = df_national['tesla_bev'] / df_national['total_bev']

# Load events
events_df = pd.read_csv(os.path.join(RAW_DIR, 'events.csv'), parse_dates=['date'])

fig, ax1 = plt.subplots(figsize=(14, 6))

quarters = df_national['quarter'].astype(str)
x = range(len(quarters))

ax1.bar(x, df_national['total_bev'], label='Total BEV', color='steelblue', alpha=0.6)
ax1.bar(x, df_national['tesla_bev'], label='Tesla BEV', color='tomato', alpha=0.8)
ax1.set_ylabel('Vehicle Registrations')
ax1.set_xticks(x)
ax1.set_xticklabels(quarters, rotation=45, ha='right', fontsize=8)

ax2 = ax1.twinx()
ax2.plot(x, df_national['tesla_share'], color='darkred', linewidth=2, label='Tesla Share')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_ylabel('Tesla Market Share')

# Add event lines
for _, row in events_df.iterrows():
    q = pd.Period(row['date'], 'Q').strftime('%YQ%q')
    if q in quarters.values:
        idx = list(quarters.values).index(q)
        ax1.axvline(x=idx, color='gray', linestyle='--', alpha=0.7)
        ax1.text(idx, ax1.get_ylim()[1]*0.85, row['event'],
                rotation=90, fontsize=7, color='gray')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_title('EV Registrations and Tesla Market Share (2020–2024)\n14-State Sample')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'tesla_market_share.png'), dpi=150)
plt.show()

This is a strong chart and it's already telling a clear story. Here's what you're seeing:
The headline finding: Tesla's market share has been declining since 2021 even as total BEV registrations keep growing. That's your central thesis — the EV market is expanding but Tesla isn't capturing the new growth proportionally.
Specific things to note:

Tesla share peaked around 72% in early 2021 and has steadily declined to around 55% by 2024
Total BEV volume roughly 10x'd from 2020 to 2024 — the market exploded
The IRA (Aug 2022) coincides with accelerated total BEV growth but Tesla share keeps sliding
The tariff announcement (May 2024) doesn't visibly reverse the share decline — important finding
2026Q1 looks like an incomplete quarter, we'll trim that

This chart alone supports your strategic argument: tariffs are a temporary shield but the competitive pressure is structural, not just from Chinese EVs but from domestic competitors filling the expanding market.

In [ ]:
# Drop incomplete 2026Q1
df_national = df_national[df_national['quarter'].astype(str) < '2026Q1']
df_pivot_clean = df_pivot[df_pivot['quarter'].astype(str) < '2026Q1']

print("Cleaned date range:", df_national['quarter'].min(), "to", df_national['quarter'].max())
print("Shape:", df_national.shape)

df_pivot_clean.to_csv(os.path.join(CLEAN_DIR, 'ev_registrations_clean.csv'), index=False)
print("Resaved without incomplete quarter.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Load trends competition data
df_trends = pd.read_csv(os.path.join(RAW_DIR, 'trends_competition.csv'), index_col=0, parse_dates=True)
events_df = pd.read_csv(os.path.join(RAW_DIR, 'events.csv'), parse_dates=['date'])

# Resample to quarterly
df_trends_q = df_trends.resample('Q').mean()
df_trends_q.index = df_trends_q.index.to_period('Q')

# Filter to 2020Q1 onwards
df_trends_q = df_trends_q[df_trends_q.index >= pd.Period('2020Q1')]

fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot Tesla share on left axis
quarters_str = df_national['quarter'].astype(str).values
x = range(len(quarters_str))
ax1.plot(x, df_national['tesla_share'] * 100, color='tomato', linewidth=2.5, label='Tesla Market Share %')
ax1.set_ylabel('Tesla Market Share (%)')
ax1.set_ylim(50, 80)

# Plot search trends on right axis
ax2 = ax1.twinx()
trend_quarters = df_trends_q.index.astype(str)

for col, color in zip(['Tesla', 'BYD electric', 'Rivian', 'Lucid Motors'],
                       ['steelblue', 'orange', 'green', 'purple']):
    if col in df_trends_q.columns:
        ax2.plot(range(len(trend_quarters)), df_trends_q[col],
                linestyle='--', linewidth=1.5, label=f'{col} (search)', color=color, alpha=0.7)

ax2.set_ylabel('Google Search Interest (0-100)')

# Event lines
for _, row in events_df.iterrows():
    q = pd.Period(row['date'], 'Q').strftime('%YQ%q')
    if q in quarters_str:
        idx = list(quarters_str).index(q)
        ax1.axvline(x=idx, color='gray', linestyle='--', alpha=0.6)
        ax1.text(idx, 79, row['event'], rotation=90, fontsize=7, color='gray')

ax1.set_xticks(x)
ax1.set_xticklabels(quarters_str, rotation=45, ha='right', fontsize=8)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left', fontsize=8)

ax1.set_title('Tesla Market Share vs. EV Brand Search Interest (2020–2025)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'tesla_share_vs_trends.png'), dpi=150)
plt.show()

Key findings visible right now:

Tesla search interest (blue dashed) stays high and relatively flat — brand awareness isn't the problem
Tesla market share (red solid) keeps declining despite sustained search interest — consumers are interested but buying competitors instead
Rivian (green) search interest grows meaningfully from 2022 onward, tracking closely with Tesla's share decline
BYD and Lucid are essentially flat near zero — Chinese EVs have no US search presence yet

The strategic argument this supports: Tesla's share erosion isn't from Chinese EV competition directly — it's from domestic competitors like Rivian filling the expanding market. The tariff is protecting Tesla from a threat that hasn't arrived yet in the US consumer market.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Load trends demand data
df_demand = pd.read_csv(os.path.join(RAW_DIR, 'trends_demand.csv'), index_col=0, parse_dates=True)
df_competition = pd.read_csv(os.path.join(RAW_DIR, 'trends_competition.csv'), index_col=0, parse_dates=True)

# Tariff announcement date
tariff_date = pd.Timestamp('2024-05-14')

# Create a window: 12 months before and after
window_start = tariff_date - pd.DateOffset(months=12)
window_end = tariff_date + pd.DateOffset(months=12)

df_window = df_competition[(df_competition.index >= window_start) & 
                            (df_competition.index <= window_end)].copy()

df_window['post_tariff'] = (df_window.index >= tariff_date).astype(int)
df_window['days_from_tariff'] = (df_window.index - tariff_date).days

# Plot event study
fig, ax = plt.subplots(figsize=(14, 6))

for col, color in zip(['Tesla', 'BYD electric', 'Rivian'],
                       ['tomato', 'orange', 'steelblue']):
    ax.plot(df_window.index, df_window[col], label=col, linewidth=2)

ax.axvline(x=tariff_date, color='black', linestyle='--', linewidth=2, label='Biden 100% Tariff (May 2024)')
ax.axvspan(window_start, tariff_date, alpha=0.05, color='green', label='Pre-tariff window')
ax.axvspan(tariff_date, window_end, alpha=0.05, color='red', label='Post-tariff window')

# Add pre/post means for Tesla
pre_mean = df_window[df_window['post_tariff']==0]['Tesla'].mean()
post_mean = df_window[df_window['post_tariff']==1]['Tesla'].mean()
ax.axhline(y=pre_mean, color='tomato', linestyle=':', alpha=0.7)
ax.axhline(y=post_mean, color='tomato', linestyle='-.', alpha=0.7)
ax.text(window_start, pre_mean+1, f'Tesla pre-mean: {pre_mean:.1f}', fontsize=8, color='tomato')
ax.text(tariff_date, post_mean+1, f'Tesla post-mean: {post_mean:.1f}', fontsize=8, color='tomato')

ax.set_title('EV Brand Search Interest: 12 Months Before and After Biden Tariff (May 2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Google Search Interest (0-100)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'tariff_event_study.png'), dpi=150)
plt.show()

# Print summary stats
print("=== Tariff Event Study: Tesla Search Interest ===")
print(f"Pre-tariff mean:  {pre_mean:.2f}")
print(f"Post-tariff mean: {post_mean:.2f}")
print(f"Change:           {post_mean - pre_mean:+.2f}")
print(f"% Change:         {((post_mean - pre_mean) / pre_mean) * 100:+.1f}%")

The numbers:

Tesla search interest went from 64.6 pre-tariff to 68.7 post-tariff — a +6.3% increase
BYD remains flat at zero throughout — no US consumer awareness before or after the tariff
Rivian spiked around early 2024 then settled back down

What this means for your argument:
The tariff coincided with a modest uptick in Tesla search interest, but it's hard to attribute causally because Tesla search is noisy week-to-week. More importantly, BYD was already at zero before the tariff — meaning the tariff protected Tesla from a threat that had no US consumer presence yet. That's the nuanced finding: the tariff was preemptive, not reactive.
One important caveat to note in your writeup: The Rivian spike in early 2024 (before the tariff) likely reflects the R2 announcement in March 2024, not tariff effects. That's worth calling out.

In [ ]:
# Load FSD trends
df_fsd = pd.read_csv(os.path.join(RAW_DIR, 'trends_fsd.csv'), index_col=0, parse_dates=True)

# FSD key events
fsd_events = {
    'FSD v12 Release': '2024-03-01',
    'FSD Unsupervised Launch': '2024-10-01',
    'FSD v11 Release': '2023-03-01'
}

fig, ax = plt.subplots(figsize=(14, 6))

for col, color in zip(['Tesla FSD', 'Tesla self driving', 'Tesla autopilot'],
                       ['tomato', 'steelblue', 'green']):
    ax.plot(df_fsd.index, df_fsd[col], label=col, linewidth=1.5)

for label, date in fsd_events.items():
    ax.axvline(x=pd.Timestamp(date), color='gray', linestyle='--', alpha=0.8)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1]*0.85, label,
            rotation=90, fontsize=8, color='gray')

ax.set_title('Tesla Autonomy Search Interest (2020–2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Google Search Interest (0-100)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fsd_search_interest.png'), dpi=150)
plt.show()

# Pre/post FSD v12 comparison
fsd_v12 = pd.Timestamp('2024-03-01')
pre_fsd = df_fsd[df_fsd.index < fsd_v12]['Tesla FSD'].mean()
post_fsd = df_fsd[df_fsd.index >= fsd_v12]['Tesla FSD'].mean()

print("=== FSD v12 Impact on Search Interest ===")
print(f"Pre-FSD v12 mean:  {pre_fsd:.2f}")
print(f"Post-FSD v12 mean: {post_fsd:.2f}")
print(f"Change:            {post_fsd - pre_fsd:+.2f}")
print(f"% Change:          {((post_fsd - pre_fsd) / pre_fsd) * 100:+.1f}%")

The numbers:

FSD search interest jumped +166% after FSD v12 release — from 12.5 to 33.3
The v12 spike hits 100 (the index maximum) — that's the single biggest search event in the entire dataset
FSD Unsupervised Launch in October 2024 triggers another visible spike
All three autonomy terms move together at each release, confirming it's a real signal not noise

What this means:
FSD releases generate massive consumer attention — far more than any other event in your dataset including the tariff announcement. This is your clearest evidence that autonomy is Tesla's most differentiated asset and strongest demand signal.
The strategic conclusion this supports:
While Tesla's market share erodes from domestic competition and the tariff provides only modest brand lift, FSD creates demand spikes that no competitor can replicate. That's the moat.

In [ ]:
df_trends = pd.read_csv(os.path.join(RAW_DIR, 'trends_competition.csv'), index_col=0, parse_dates=True)
print("Weekly observations:", len(df_trends))
print("Date range:", df_trends.index.min(), "to", df_trends.index.max())
print("BYD electric - non-zero weeks:", (df_trends['BYD electric'] > 0).sum())
print("BYD electric - mean:", df_trends['BYD electric'].mean())
print("NIO car - non-zero weeks:", (df_trends['NIO car'] > 0).sum())

In [ ]:
# Check what makes appear in your registration data
df_all = pd.concat([pd.read_csv(os.path.join(RAW_DIR, f), low_memory=False) for f in files])

# Look for any Chinese EV brands
chinese_brands = ['BYD', 'NIO', 'XPENG', 'LI AUTO', 'ZEEKR', 'GEELY', 'SAIC']
for brand in chinese_brands:
    count = df_all[df_all['Vehicle Make'].str.upper().str.contains(brand, na=False)]['Vehicle Count'].sum()
    print(f"{brand}: {count} vehicles")

# Also print top 10 makes for context
print("\nTop 10 makes in sample:")
print(df_all.groupby('Vehicle Make')['Vehicle Count'].sum().sort_values(ascending=False).head(10))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Registration data by make - BEV only, top makes
df_bev_makes = df_all[df_all['Drivetrain Type'] == 'BEV'].copy()
make_counts = df_bev_makes.groupby('Vehicle Make')['Vehicle Count'].sum().sort_values(ascending=False)

# Keep top 8 + Chinese brands
top_makes = make_counts.head(8).index.tolist()
chinese = ['BYD']
all_makes = top_makes + [b for b in chinese if b not in top_makes]
make_counts_plot = make_counts[all_makes]

# Trends averages
trends_means = df_trends.mean()
brands = ['Tesla', 'BYD electric', 'NIO car', 'Rivian', 'Lucid Motors']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0A0A0A')

# Left: registrations
colors = ['#E31937' if 'TESLA' in m else '#333333' for m in make_counts_plot.index]
ax1.barh(make_counts_plot.index, make_counts_plot.values / 1e6, color=colors)
ax1.set_facecolor('#0A0A0A')
ax1.set_xlabel('Total Registrations (millions)', color='white')
ax1.set_title('BEV Registrations by Make\n14-State Sample', color='white', fontsize=11)
ax1.tick_params(colors='white')
ax1.spines['bottom'].set_color('#444444')
ax1.spines['left'].set_color('#444444')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: search interest
colors2 = ['#E31937' if b == 'Tesla' else '#555555' if b in ['Rivian', 'Lucid Motors'] else '#222222' for b in brands]
ax2.bar(brands, trends_means[brands], color=colors2)
ax2.set_facecolor('#0A0A0A')
ax2.set_ylabel('Avg Search Interest (0-100)', color='white')
ax2.set_title('Google Trends Avg Search Interest\nUS, 2020-2024', color='white', fontsize=11)
ax2.tick_params(colors='white')
ax2.spines['bottom'].set_color('#444444')
ax2.spines['left'].set_color('#444444')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.xticks(rotation=15, ha='right')

fig.suptitle('Chinese EVs: Zero Market Presence, Zero Consumer Awareness', 
             color='white', fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'chinese_ev_zero_presence.png'), dpi=150, bbox_inches='tight', facecolor='#0A0A0A')
plt.show()

In [ ]:
import os
print(os.path.join(FIG_DIR, 'chinese_ev_zero_presence.png'))
import os.path
print(os.path.exists(os.path.join(FIG_DIR, 'chinese_ev_zero_presence.png')))

In [ ]:
# BEV registrations by make, filtered to 2020 onwards, excluding Tesla
df_bev_recent = df_bev[df_bev['quarter'].astype(str) >= '2020Q1'].copy()
df_competitors = df_bev_recent[df_bev_recent['Vehicle Make'].str.upper() != 'TESLA']

# Total registrations by make
competitor_totals = df_competitors.groupby('Vehicle Make')['Vehicle Count'].sum().sort_values(ascending=False)
print("Top 10 competitors by total BEV registrations 2020-2025:")
print(competitor_totals.head(10))

# Growth rate by make - compare 2020 vs 2024
df_2020 = df_bev_recent[df_bev_recent['quarter'].astype(str).str.startswith('2020')]
df_2024 = df_bev_recent[df_bev_recent['quarter'].astype(str).str.startswith('2024')]

totals_2020 = df_2020[df_2020['Vehicle Make'].str.upper() != 'TESLA'].groupby('Vehicle Make')['Vehicle Count'].sum()
totals_2024 = df_2024[df_2024['Vehicle Make'].str.upper() != 'TESLA'].groupby('Vehicle Make')['Vehicle Count'].sum()

growth = ((totals_2024 - totals_2020) / totals_2020 * 100).sort_values(ascending=False)
print("\nTop 10 competitors by growth rate 2020 vs 2024:")
print(growth.head(10))

In [ ]:
# Filter out 2026 and NaT before plotting
df_comp_yearly = df_bev_recent[
    df_bev_recent['Vehicle Make'].str.upper() != 'TESLA'
].copy()
df_comp_yearly['year'] = df_comp_yearly['quarter'].astype(str).str[:4]

# Keep only valid years 2020-2025
valid_years = ['2020', '2021', '2022', '2023', '2024', '2025']
df_comp_yearly = df_comp_yearly[df_comp_yearly['year'].isin(valid_years)]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Aggregate non-Tesla BEV by make and year
df_comp_yearly = df_bev_recent[
    df_bev_recent['Vehicle Make'].str.upper() != 'TESLA'
].copy()
df_comp_yearly['year'] = df_comp_yearly['quarter'].astype(str).str[:4]

# Focus on top competitors by total volume
top_makes = ['NISSAN', 'CHEVROLET', 'FORD', 'HYUNDAI', 'KIA', 'RIVIAN', 'VOLKSWAGEN', 'BMW']

df_top = df_comp_yearly[df_comp_yearly['Vehicle Make'].str.upper().isin(top_makes)]
yearly_by_make = df_top.groupby(['year', 'Vehicle Make'])['Vehicle Count'].sum().unstack(fill_value=0)

# Highlight colors
highlight = {'FORD': '#CC0000', 'HYUNDAI': '#F59E0B', 'KIA': '#F59E0B'}
colors = {make: highlight.get(make, '#2A2A2A') for make in yearly_by_make.columns}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0A0A0A')

# LEFT: Stacked bar by year showing competitor growth
bottom = np.zeros(len(yearly_by_make))
for make in yearly_by_make.columns:
    color = '#CC0000' if make == 'FORD' else '#D97706' if make in ['HYUNDAI', 'KIA'] else '#2A2A2A'
    ax1.bar(yearly_by_make.index, yearly_by_make[make] / 1e6,
            bottom=bottom, label=make, color=color, alpha=0.9)
    bottom += yearly_by_make[make].values / 1e6

ax1.set_facecolor('#0A0A0A')
ax1.set_title('Non-Tesla BEV Registrations by Make\n14-State Sample, 2020–2025',
              color='white', fontsize=11, pad=10)
ax1.set_xlabel('Year', color='white')
ax1.set_ylabel('Registrations (millions)', color='white')
ax1.tick_params(colors='white')
ax1.spines['bottom'].set_color('#444444')
ax1.spines['left'].set_color('#444444')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
legend = ax1.legend(loc='upper left', fontsize=7, facecolor='#1A1A1A',
                    labelcolor='white', framealpha=0.8)

# RIGHT: Total volume bar chart with highlights
makes_order = competitor_totals.head(8).index.tolist()
totals = competitor_totals.head(8).values / 1e6
bar_colors = ['#CC0000' if m == 'FORD' else '#D97706' if m in ['HYUNDAI', 'KIA'] else '#2A2A2A'
              for m in makes_order]

bars = ax2.barh(makes_order[::-1], totals[::-1], color=bar_colors[::-1], alpha=0.9)
ax2.set_facecolor('#0A0A0A')
ax2.set_title("Total BEV Registrations 2020–2025\nFord + Hyundai/Kia = Tesla's Core Threat",
              color='white', fontsize=11, pad=10)
ax2.set_xlabel('Total Registrations (millions)', color='white')
ax2.tick_params(colors='white')
ax2.spines['bottom'].set_color('#444444')
ax2.spines['left'].set_color('#444444')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Annotation arrows
ax2.annotate('Truck segment\n(no Tesla response)', xy=(1.19, 1), xytext=(1.5, 1),
             color='#CC0000', fontsize=8,
             arrowprops=dict(arrowstyle='->', color='#CC0000', lw=1.2))
ax2.annotate('Sub-$40k sedan\n(Model 3 threat)', xy=(0.877, 3), xytext=(1.2, 3.4),
             color='#D97706', fontsize=8,
             arrowprops=dict(arrowstyle='->', color='#D97706', lw=1.2))

# Legend
ford_patch = mpatches.Patch(color='#CC0000', label='Ford — Truck segment gap')
hk_patch = mpatches.Patch(color='#D97706', label='Hyundai/Kia — Sub-$40k sedan gap')
other_patch = mpatches.Patch(color='#2A2A2A', label='Other competitors')
ax2.legend(handles=[ford_patch, hk_patch, other_patch],
           loc='lower right', fontsize=8, facecolor='#1A1A1A',
           labelcolor='white', framealpha=0.8)

plt.suptitle("Tesla's Primary Domestic Competitors: Ford and Hyundai/Kia",
             color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'competitor_analysis.png'),
            dpi=150, bbox_inches='tight', facecolor='#0A0A0A')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Filter and prep data
df_comp_clean = df_bev_recent[
    (df_bev_recent['Vehicle Make'].str.upper() != 'TESLA')
].copy()
df_comp_clean['year'] = df_comp_clean['quarter'].astype(str).str[:4]
df_comp_clean = df_comp_clean[df_comp_clean['year'].isin(['2020','2021','2022','2023','2024','2025'])]

top_makes = ['NISSAN', 'CHEVROLET', 'FORD', 'HYUNDAI', 'KIA', 'RIVIAN', 'VOLKSWAGEN', 'BMW']
df_top = df_comp_clean[df_comp_clean['Vehicle Make'].str.upper().isin(top_makes)]

# Yearly stacked data
yearly_by_make = df_top.groupby(['year', 'Vehicle Make'])['Vehicle Count'].sum().unstack(fill_value=0)

# Total by make
totals_by_make = df_top.groupby('Vehicle Make')['Vehicle Count'].sum().sort_values(ascending=False)
makes_order = totals_by_make.index.tolist()
totals = totals_by_make.values / 1e6

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0A0A0A')

# LEFT: stacked bar
bottom = np.zeros(len(yearly_by_make))
for make in yearly_by_make.columns:
    color = '#CC0000' if make == 'FORD' else '#D97706' if make in ['HYUNDAI', 'KIA'] else '#2A2A2A'
    ax1.bar(yearly_by_make.index, yearly_by_make[make] / 1e6,
            bottom=bottom, label=make, color=color, alpha=0.9)
    bottom += yearly_by_make[make].values / 1e6

ax1.set_facecolor('#0A0A0A')
ax1.set_title('Non-Tesla BEV Registrations by Make\n14-State Sample, 2020–2025', color='white', fontsize=11)
ax1.set_xlabel('Year', color='white')
ax1.set_ylabel('Registrations (millions)', color='white')
ax1.tick_params(colors='white')
ax1.spines['bottom'].set_color('#444444')
ax1.spines['left'].set_color('#444444')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_xticks(['2020','2021','2022','2023','2024','2025'])
ax1.legend(loc='upper left', fontsize=7, facecolor='#1A1A1A', labelcolor='white', framealpha=0.8)

# RIGHT: horizontal bar
bar_colors = ['#CC0000' if m == 'FORD' else '#D97706' if m in ['HYUNDAI', 'KIA'] else '#2A2A2A' for m in makes_order]
ax2.barh(makes_order[::-1], totals[::-1], color=bar_colors[::-1], alpha=0.9)
ax2.set_facecolor('#0A0A0A')
ax2.set_title("Total BEV Registrations 2020–2025\nFord + Hyundai/Kia = Tesla's Core Threat", color='white', fontsize=11)
ax2.set_xlabel('Total Registrations (millions)', color='white')
ax2.tick_params(colors='white')
ax2.spines['bottom'].set_color('#444444')
ax2.spines['left'].set_color('#444444')
ax2.spines['top'].set_v

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#0A0A0A')
ax.set_facecolor('#0A0A0A')
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# Title
ax.text(7, 7.5, 'OLS Regression with Two-Way Fixed Effects', 
        ha='center', va='center', fontsize=14, color='white', 
        fontweight='bold', fontfamily='Arial')
ax.text(7, 7.0, 'Dependent Variable: Tesla BEV Market Share (state i, quarter t)',
        ha='center', va='center', fontsize=10, color='#999999', fontfamily='Arial')

# Outcome box
ax.add_patch(plt.Rectangle((5.5, 3.0), 3, 1.5, 
             facecolor='#E31937', edgecolor='#E31937', linewidth=0))
ax.text(7, 3.95, 'Tesla Market Share', ha='center', va='center', 
        fontsize=11, color='white', fontweight='bold', fontfamily='Arial')
ax.text(7, 3.45, 'state i × quarter t', ha='center', va='center',
        fontsize=9, color='#FFAAAA', fontfamily='Arial')

# Predictor boxes — significant
sig_vars = [
    ('Gas Price\n($/gallon)', '−0.099*', 1.0, 5.5),
    ('Post-IRA\n(Aug 2022+)', '−0.059**', 1.0, 3.75),
    ('Post-Tariff\n(May 2024+)', '−0.089**', 1.0, 2.0),
]
for label, coef, x, y in sig_vars:
    ax.add_patch(plt.Rectangle((x, y), 2.2, 1.0,
                 facecolor='#1A1A1A', edgecolor='#E31937', linewidth=1.5))
    ax.text(x+1.1, y+0.65, label, ha='center', va='center',
            fontsize=9, color='white', fontfamily='Arial')
    ax.text(x+1.1, y+0.2, coef, ha='center', va='center',
            fontsize=10, color='#E31937', fontweight='bold', fontfamily='Arial')
    ax.annotate('', xy=(5.5, 3.75), xytext=(3.2, y+0.5),
                arrowprops=dict(arrowstyle='->', color='#E31937', lw=1.5))

# Predictor box — not significant
ax.add_patch(plt.Rectangle((1.0, 0.3), 2.2, 1.0,
             facecolor='#1A1A1A', edgecolor='#444444', linewidth=1))
ax.text(2.1, 0.95, 'Rivian Search\n(attention proxy)', ha='center', va='center',
        fontsize=9, color='#999999', fontfamily='Arial')
ax.text(2.1, 0.5, '+0.241 (n.s.)', ha='center', va='center',
        fontsize=10, color='#666666', fontweight='bold', fontfamily='Arial')
ax.annotate('', xy=(5.5, 3.5), xytext=(3.2, 0.8),
            arrowprops=dict(arrowstyle='->', color='#444444', lw=1.0, linestyle='dashed'))

# Fixed effects boxes
fe_items = [
    ('State Fixed Effects\n(14 dummies)', '← Absorbs stable cross-state\ndifferences', 9.5, 5.2),
    ('Quarter Fixed Effects\n(24 dummies)', '← Absorbs common\ntime shocks', 9.5, 3.2),
]
for label, desc, x, y in fe_items:
    ax.add_patch(plt.Rectangle((x, y), 4.0, 1.2,
                 facecolor='#111111', edgecolor='#F59E0B', linewidth=1.5))
    ax.text(x+2.0, y+0.82, label, ha='center', va='center',
            fontsize=9, color='#F59E0B', fontweight='bold', fontfamily='Arial')
    ax.text(x+2.0, y+0.32, desc, ha='center', va='center',
            fontsize=8, color='#999999', fontfamily='Arial')
    ax.annotate('', xy=(8.5, 3.75), xytext=(9.5, y+0.6),
                arrowprops=dict(arrowstyle='->', color='#F59E0B', lw=1.5))

# What's being identified label
ax.add_patch(plt.Rectangle((3.5, 0.2), 4.5, 0.9,
             facecolor='#0F0F0F', edgecolor='#444444', linewidth=1))
ax.text(5.75, 0.65, 'Model identifies:', ha='center', va='center',
        fontsize=8.5, color='#999999', fontfamily='Arial')
ax.text(5.75, 0.38, 'Within-state variation in Tesla share over time',
        ha='center', va='center', fontsize=9, color='white', fontweight='bold', fontfamily='Arial')

# Stats box
ax.add_patch(plt.Rectangle((9.5, 0.2), 4.0, 2.5,
             facecolor='#161616', edgecolor='#444444', linewidth=1))
ax.text(11.5, 2.45, 'MODEL FIT', ha='center', va='center',
        fontsize=8, color='#E31937', fontfamily='Arial',
        fontweight='bold')
stats = [
    ('R²', '0.863'),
    ('Adj. R²', '0.844'),
    ('N', '298'),
    ('Specification', 'Two-way FE OLS'),
    ('Unit of obs.', 'State × Quarter'),
]
for j, (label, val) in enumerate(stats):
    ax.text(10.0, 2.1 - j*0.42, label, ha='left', va='center',
            fontsize=8.5, color='#999999', fontfamily='Arial')
    ax.text(13.4, 2.1 - j*0.42, val, ha='right', va='center',
            fontsize=8.5, color='white', fontfamily='Arial', fontweight='bold')

# Legend
sig_patch = mpatches.Patch(facecolor='#1A1A1A', edgecolor='#E31937', label='Significant predictor (p<0.05)')
ns_patch = mpatches.Patch(facecolor='#1A1A1A', edgecolor='#444444', label='Not significant')
fe_patch = mpatches.Patch(facecolor='#111111', edgecolor='#F59E0B', label='Fixed effects (controls)')
ax.legend(handles=[sig_patch, ns_patch, fe_patch], loc='lower left',
          fontsize=8, facecolor='#1A1A1A', labelcolor='white', 
          framealpha=0.9, bbox_to_anchor=(0.0, 0.0))

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'model_diagram.png'), 
            dpi=150, bbox_inches='tight', facecolor='#0A0A0A')
plt.show()